In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# EE 678 — Daubechies Wavelets & Differential Equations
# Complete figure generation — single cell
# Generates: fig_wavelet_gallery, fig_sparsity_pattern, fig_eigenvalues,
#            fig_ode_convergence, fig_ode_solution, fig_heat_convergence,
#            fig_heat_solution, fig_work_accuracy, fig_symbol_analysis
# ═══════════════════════════════════════════════════════════════════════════════

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import scipy
from scipy import linalg
import pywt
import warnings, os
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Serif', 'font.size': 11,
    'axes.titlesize': 12, 'axes.labelsize': 11, 'legend.fontsize': 9,
    'figure.dpi': 150, 'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'axes.spines.top': False, 'axes.spines.right': False, 'lines.linewidth': 1.8,
})
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b']
WNAMES = ['db1', 'db2', 'db3', 'db4']
LABELS = [r'$\mathrm{Db}_1$ (Haar)', r'$\mathrm{Db}_2$', r'$\mathrm{Db}_3$', r'$\mathrm{Db}_4$']

# ─── HELPERS ──────────────────────────────────────────────────────────────────

def get_phi_psi(wname, levels=12):
    phi, psi, x = pywt.Wavelet(wname).wavefun(level=levels)
    return x, phi, psi

def compute_connection_coeffs(wname):
    """Lambda^(1,1)_k via eigenvector of autocorrelation matrix."""
    w   = pywt.Wavelet(wname)
    h   = np.array(w.filter_bank[0]) * np.sqrt(2)
    N   = len(h) // 2
    sup = 2 * N - 1
    size = 2 * sup - 1
    idx  = np.arange(size) - (sup - 1)
    acorr = np.correlate(h, h, mode='full')
    acorr_idx = np.arange(len(acorr)) - (len(acorr) // 2)
    A = np.zeros((size, size))
    for i, ki in enumerate(idx):
        for j, kj in enumerate(idx):
            shift = 2 * (ki - kj)
            m = np.where(acorr_idx == shift)[0]
            if len(m): A[i, j] = 2.0 * acorr[m[0]]
    eigvals, eigvecs = np.linalg.eig(A)
    v = np.real(eigvecs[:, np.argmin(np.abs(eigvals - 1.0))])
    x_c, _, _ = get_phi_psi(wname)
    dx = (x_c[-1] - x_c[0]) / (len(x_c) - 1)
    phi_vals = pywt.Wavelet(wname).wavefun(level=12)[0]
    dphi = np.gradient(phi_vals, dx)
    lam0_num = np.trapz(dphi * dphi, dx=dx)
    mid = size // 2
    scale = lam0_num / v[mid] if abs(v[mid]) > 1e-12 else 1.0
    v *= scale
    return {k: v[i] for i, k in enumerate(idx)}

def build_stiffness(wname, J):
    cc   = compute_connection_coeffs(wname)
    M    = 2**J;  Mint = M - 1;  h = 1.0 / M
    A    = np.zeros((Mint, Mint));  scale = 4.0**J
    for i in range(Mint):
        for j in range(Mint):
            k = i - j
            if k in cc: A[i, j] = scale * cc[k]
    return A, h

def build_fd(M):
    h = 1.0 / M;  Mint = M - 1
    d = np.ones(Mint) * 2 / h**2
    o = -np.ones(Mint - 1) / h**2
    return np.diag(d) + np.diag(o, 1) + np.diag(o, -1), h

def get_A(wname, J):
    try:    return build_stiffness(wname, J)
    except: return build_fd(2**J)

def solve_ode(wname, J):
    A, h = get_A(wname, J)
    M = 2**J;  x = np.linspace(h, 1-h, M-1)
    u = np.linalg.solve(A, np.pi**2 * np.sin(np.pi * x))
    err = np.sqrt(np.trapz((u - np.sin(np.pi * x))**2, x))
    return x, u, err

def solve_heat_cn(wname, J, T=0.1, dt=1e-3):
    A, h = get_A(wname, J)
    M = 2**J;  x = np.linspace(h, 1-h, M-1)
    u = np.sin(np.pi * x)
    I = np.eye(M-1)
    LHS = I + 0.5*dt*A;  RHS_op = I - 0.5*dt*A
    for _ in range(int(T/dt)):
        u = np.linalg.solve(LHS, RHS_op @ u)
    t_act = int(T/dt)*dt
    u_ex  = np.exp(-np.pi**2 * t_act) * np.sin(np.pi * x)
    return x, u, np.sqrt(np.trapz((u - u_ex)**2, x))

# ─── FIG 1: Wavelet Gallery ───────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(13, 5))
fig.suptitle(r'Daubechies $\mathrm{Db}_N$: Scaling Functions $\phi$ and Wavelets $\psi$',
             fontsize=13, fontweight='bold', y=1.01)
for col, (wn, lb, c) in enumerate(zip(WNAMES, LABELS, COLORS)):
    x, phi, psi = get_phi_psi(wn)
    N = len(pywt.Wavelet(wn).filter_bank[0]) // 2
    for row, (func, ylabel) in enumerate([(phi, r'$\phi(x)$'), (psi, r'$\psi(x)$')]):
        ax = axes[row, col]
        ax.plot(x, func, color=c, lw=2); ax.fill_between(x, func, alpha=0.12, color=c)
        ax.axhline(0, color='k', lw=0.6, ls='--'); ax.set_xlim(x[0], x[-1])
        ax.tick_params(labelsize=8)
        if col == 0: ax.set_ylabel(ylabel)
        if row == 0: ax.set_title(lb, fontsize=11, fontweight='bold')
        if row == 1: ax.set_xlabel('$x$', fontsize=10)
    axes[0, col].text(0.97, 0.92, fr'supp$=[0,{2*N-1}]$', transform=axes[0,col].transAxes,
                      ha='right', va='top', fontsize=7.5, color=c)
    axes[1, col].text(0.97, 0.08, fr'{N} van. moment{"s" if N>1 else ""}',
                      transform=axes[1,col].transAxes, ha='right', va='bottom', fontsize=7.5, color=c)
fig.tight_layout(); plt.savefig('fig_wavelet_gallery.png'); plt.show()
print('✓ fig_wavelet_gallery.png')

# ─── FIG 2: Sparsity Patterns ─────────────────────────────────────────────────
J_spy = 5
fig, axes = plt.subplots(1, 4, figsize=(13, 3.8))
fig.suptitle(fr'Sparsity Patterns of Stiffness Matrix $A$, $J={J_spy}$ ($M={2**J_spy-1}$ DOFs)',
             fontsize=12, fontweight='bold')
for col, (wn, lb, c) in enumerate(zip(WNAMES, LABELS, COLORS)):
    A, _ = get_A(wn, J_spy)
    A_p  = np.abs(A); A_p[A_p < 1e-10*A_p.max()] = 0
    axes[col].spy(A_p, markersize=3, color=c)
    axes[col].set_title(lb, fontsize=11, fontweight='bold')
    N   = len(pywt.Wavelet(wn).filter_bank[0]) // 2
    axes[col].set_xlabel(fr'NNZ={np.sum(A_p>0)}, bw={max(1,2*N-2)}', fontsize=8)
fig.tight_layout(); plt.savefig('fig_sparsity_pattern.png'); plt.show()
print('✓ fig_sparsity_pattern.png')

# ─── FIG 3: Eigenvalue Spectra ────────────────────────────────────────────────
J_eig = 5;  Me = 2**J_eig - 1
lam_ex = (np.arange(1, Me+1) * np.pi)**2
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
for wn, lb, c in zip(WNAMES, LABELS, COLORS):
    A, _ = get_A(wn, J_eig)
    lam  = np.sort(np.real(np.linalg.eigvals(A))); lam = lam[lam > 0]
    n    = min(len(lam), Me)
    ax1.plot(np.arange(1, n+1), lam[:n], 'o-', color=c, ms=3, lw=1.5, label=lb)
    ax2.semilogy(np.arange(1, n+1),
                 np.abs(lam[:n]-lam_ex[:n])/lam_ex[:n]+1e-16,
                 'o-', color=c, ms=3, lw=1.5, label=lb)
ax1.plot(np.arange(1, Me+1), lam_ex, 'k--', lw=1.2, label=r'Exact $k^2\pi^2$')
ax1.set_xlabel('Mode $k$'); ax1.set_ylabel(r'$\lambda_k$')
ax1.set_title('Eigenvalue Spectrum of $-d^2/dx^2$', fontweight='bold'); ax1.legend(fontsize=8)
ax2.set_xlabel('Mode $k$'); ax2.set_ylabel('Relative eigenvalue error')
ax2.set_title('Relative Error', fontweight='bold'); ax2.legend(fontsize=8)
ax2.grid(True, which='both', alpha=0.3)
fig.suptitle(fr'Eigenvalue Analysis, $J={J_eig}$', fontweight='bold', fontsize=12)
fig.tight_layout(); plt.savefig('fig_eigenvalues.png'); plt.show()
print('✓ fig_eigenvalues.png')

# ─── FIG 4: ODE Convergence ───────────────────────────────────────────────────
J_range = range(3, 9);  M_range = [2**J for J in J_range]
errors  = {wn: [solve_ode(wn, J)[2] for J in J_range] for wn in WNAMES}
fig, ax = plt.subplots(figsize=(7, 5))
for wn, lb, c in zip(WNAMES, LABELS, COLORS):
    errs = errors[wn]
    ax.loglog(M_range, errs, 'o-', color=c, lw=2, ms=6, label=lb)
    N = len(pywt.Wavelet(wn).filter_bank[0]) // 2
    C = errs[-3] * M_range[-3]**N
    Mr = np.array([M_range[0], M_range[-1]], dtype=float)
    ax.loglog(Mr, C/Mr**N, '--', color=c, lw=1.0, alpha=0.7, label=fr'slope $-{N}$')
ax.set_xlabel('$M=2^J$'); ax.set_ylabel(r'$\|u^*-u_J\|_{L^2}$')
ax.set_title('ODE Convergence: $-u^{\prime\prime}=\\pi^2\\sin(\\pi x)$', fontweight='bold')
ax.legend(ncol=2, fontsize=8); ax.grid(True, which='both', alpha=0.3)
fig.tight_layout(); plt.savefig('fig_ode_convergence.png'); plt.show()
print('✓ fig_ode_convergence.png')

# ─── FIG 5: ODE Solution ──────────────────────────────────────────────────────
J_sol = 4
fig, (ax_s, ax_e) = plt.subplots(1, 2, figsize=(12, 4.5))
xf = np.linspace(0, 1, 500)
ax_s.plot(xf, np.sin(np.pi*xf), 'k-', lw=2.5, label=r'Exact $\sin(\pi x)$', zorder=5)
for wn, lb, c in zip(WNAMES, LABELS, COLORS):
    xi, u, _ = solve_ode(wn, J_sol)
    ax_s.plot(xi, u, 'o--', color=c, ms=4, lw=1.5, label=lb)
    ax_e.semilogy(xi, np.abs(u - np.sin(np.pi*xi))+1e-16, 'o-', color=c, ms=3, lw=1.5, label=lb)
ax_s.set_xlabel('$x$'); ax_s.set_ylabel('$u(x)$')
ax_s.set_title(fr'Solutions at $J={J_sol}$', fontweight='bold'); ax_s.legend(fontsize=8)
ax_e.set_xlabel('$x$'); ax_e.set_ylabel('Pointwise error')
ax_e.set_title('Pointwise Error', fontweight='bold'); ax_e.legend(fontsize=8)
ax_e.grid(True, which='both', alpha=0.3)
fig.suptitle('Sturm–Liouville BVP: Solutions and Errors', fontweight='bold', fontsize=13)
fig.tight_layout(); plt.savefig('fig_ode_solution.png'); plt.show()
print('✓ fig_ode_solution.png')

# ─── FIG 6: Heat Convergence ──────────────────────────────────────────────────
Jh_range = range(3, 8);  Mh_range = [2**J for J in Jh_range]
h_errs   = {wn: [solve_heat_cn(wn, J, dt=1e-4)[2] for J in Jh_range] for wn in WNAMES}
DT_RANGE = [5e-2, 2e-2, 1e-2, 5e-3, 2e-3, 1e-3]
t_errs   = [solve_heat_cn('db4', 6, dt=dt)[2] for dt in DT_RANGE]
fig, (ax_sp, ax_tm) = plt.subplots(1, 2, figsize=(12, 5))
for wn, lb, c in zip(WNAMES, LABELS, COLORS):
    errs = h_errs[wn]
    ax_sp.loglog(Mh_range, errs, 'o-', color=c, lw=2, ms=6, label=lb)
    N  = len(pywt.Wavelet(wn).filter_bank[0]) // 2
    C  = errs[-2] * Mh_range[-2]**N
    Mr = np.array([Mh_range[0], Mh_range[-1]], dtype=float)
    ax_sp.loglog(Mr, C/Mr**N, '--', color=c, lw=0.9, alpha=0.6)
ax_sp.set_xlabel('$M=2^J$'); ax_sp.set_ylabel(r'$L^2$ error at $T=0.1$')
ax_sp.set_title('Spatial Convergence (CN, $\\Delta t=10^{-4}$)', fontweight='bold')
ax_sp.legend(fontsize=8); ax_sp.grid(True, which='both', alpha=0.3)
ax_tm.loglog(DT_RANGE, t_errs, 's-', color=COLORS[3], lw=2, ms=7, label=r'$\mathrm{Db}_4$, $J=6$')
C2 = t_errs[2] / DT_RANGE[2]**2
dtr = np.array([DT_RANGE[0], DT_RANGE[-1]])
ax_tm.loglog(dtr, C2*dtr**2, 'k--', lw=1.2, label='slope $+2$ (CN)')
ax_tm.set_xlabel(r'$\Delta t$'); ax_tm.set_ylabel(r'$L^2$ error at $T=0.1$')
ax_tm.set_title(r'Temporal Convergence ($\mathrm{Db}_4$, $J=6$)', fontweight='bold')
ax_tm.legend(fontsize=8); ax_tm.grid(True, which='both', alpha=0.3)
fig.suptitle('Heat Equation Convergence Study', fontweight='bold', fontsize=13)
fig.tight_layout(); plt.savefig('fig_heat_convergence.png'); plt.show()
print('✓ fig_heat_convergence.png')

# ─── FIG 7: Heat Solution Snapshots ──────────────────────────────────────────
J_hs = 5;  dt_hs = 1e-3
A_hs, h_hs = get_A('db4', J_hs)
Mint_hs    = 2**J_hs - 1
x_hs       = np.linspace(h_hs, 1-h_hs, Mint_hs)
I_hs       = np.eye(Mint_hs)
LHS_hs     = I_hs + 0.5*dt_hs*A_hs;  ROP_hs = I_hs - 0.5*dt_hs*A_hs
u_hs       = np.sin(np.pi * x_hs)
T_snaps    = [0.0, 0.05, 0.10, 0.20]
snaps      = {0.0: u_hs.copy()};  t_hs = 0.0
while t_hs < max(T_snaps) - 1e-12:
    u_hs = np.linalg.solve(LHS_hs, ROP_hs @ u_hs);  t_hs += dt_hs
    for ts in T_snaps:
        if abs(t_hs - ts) < dt_hs*0.5 and ts not in snaps:
            snaps[ts] = u_hs.copy()
xf2 = np.linspace(0, 1, 500)
fig, (ax_sn, ax_dc) = plt.subplots(1, 2, figsize=(12, 4.5))
cmap_sn = plt.cm.plasma(np.linspace(0.15, 0.85, len(T_snaps)))
for ts, c in zip(T_snaps, cmap_sn):
    if ts in snaps:
        ax_sn.plot(x_hs, snaps[ts], 'o-', color=c, ms=2.5, lw=1.8, label=fr'$t={ts}$ (num.)')
    ax_sn.plot(xf2, np.exp(-np.pi**2*ts)*np.sin(np.pi*xf2), '--', color=c, lw=1.0, alpha=0.6)
ax_sn.set_xlabel('$x$'); ax_sn.set_ylabel('$u(x,t)$')
ax_sn.set_title(r'$\mathrm{Db}_4$ Heat Equation Snapshots', fontweight='bold')
ax_sn.legend(fontsize=7, ncol=2)
T_dc = np.linspace(0, 0.25, 200)
ax_dc.plot(T_dc, np.exp(-np.pi**2*T_dc), 'k-', lw=2, label=r'Exact: $e^{-\pi^2 t}$')
t_v  = sorted(snaps.keys())
ax_dc.plot(t_v, [np.max(np.abs(snaps[t])) for t in t_v], 's', color=COLORS[3], ms=9,
           zorder=5, label=r'$\mathrm{Db}_4$ max')
ax_dc.set_xlabel('$t$'); ax_dc.set_ylabel(r'$\max_x|u|$')
ax_dc.set_title('Amplitude Decay', fontweight='bold'); ax_dc.legend(fontsize=9)
ax_dc.grid(True, alpha=0.3)
fig.suptitle('Heat Equation: Wavelet–Galerkin (Crank–Nicolson)', fontweight='bold', fontsize=13)
fig.tight_layout(); plt.savefig('fig_heat_solution.png'); plt.show()
print('✓ fig_heat_solution.png')

# ─── FIG 8: Work vs. Accuracy ─────────────────────────────────────────────────
WN_ext = ['db1','db2','db3','db4','db5','db6']
LB_ext = [fr'$\mathrm{{Db}}_{{{n}}}$' for n in range(1,7)]
CL_ext = plt.cm.tab10(np.linspace(0, 0.6, 6))
fig, ax = plt.subplots(figsize=(8, 5.5))
for wn, lb, c in zip(WN_ext, LB_ext, CL_ext):
    N   = len(pywt.Wavelet(wn).filter_bank[0]) // 2
    bw  = max(1, 2*N - 1)
    works, errs = [], []
    for J in range(2, 10):
        works.append((2**J - 1) * bw**2)
        errs.append(solve_ode(wn, J)[2])
    ax.loglog(works, errs, 'o-', color=c, lw=1.8, ms=5, label=lb)
ax.set_xlabel('Computational work ($M \\cdot b_N^2$)')
ax.set_ylabel(r'$L^2$ error'); ax.legend(ncol=2, fontsize=9)
ax.set_title('Work vs. Accuracy\n'
             r'Optimal $N^* \approx \ln(1/\varepsilon)/2$', fontweight='bold')
ax.grid(True, which='both', alpha=0.3)
ax.axhspan(1e-6, 1e-4, alpha=0.08, color='green')
ax.text(3e4, 3e-5, r'Typical target: $N=3,4$ optimal', fontsize=8, color='darkgreen')
fig.tight_layout(); plt.savefig('fig_work_accuracy.png'); plt.show()
print('✓ fig_work_accuracy.png')

# ─── FIG 9: Operator Symbol Analysis ─────────────────────────────────────────
xi_s = np.linspace(0, np.pi, 300)
fig, (ax_sy, ax_se) = plt.subplots(1, 2, figsize=(12, 4.5))
for wn, lb, c in zip(WNAMES, LABELS, COLORS):
    cc  = compute_connection_coeffs(wn)
    a   = sum(lam * np.cos(k * xi_s) for k, lam in cc.items())
    ax_sy.plot(xi_s, a, color=c, lw=2, label=lb)
    ax_se.semilogy(xi_s[1:], np.abs(a[1:]-xi_s[1:]**2)/xi_s[1:]**2+1e-16, color=c, lw=2, label=lb)
ax_sy.plot(xi_s, xi_s**2, 'k--', lw=1.5, label=r'Exact $\xi^2$')
ax_sy.set_xlabel(r'$\xi$'); ax_sy.set_ylabel(r'$a(\xi)$')
ax_sy.set_title(r'Symbol $a(\xi)$ vs. $\xi^2$', fontweight='bold'); ax_sy.legend(fontsize=8)
ax_se.set_xlabel(r'$\xi$'); ax_se.set_ylabel(r'$|a(\xi)-\xi^2|/\xi^2$')
ax_se.set_title('Relative Symbol Error', fontweight='bold'); ax_se.legend(fontsize=8)
ax_se.grid(True, which='both', alpha=0.3)
fig.suptitle('Operator Symbol: Spectral Accuracy of Db_N Galerkin',
             fontweight='bold', fontsize=12)
fig.tight_layout(); plt.savefig('fig_symbol_analysis.png'); plt.show()
print('✓ fig_symbol_analysis.png')

# ─── SUMMARY ──────────────────────────────────────────────────────────────────
figs = ['fig_wavelet_gallery.png','fig_sparsity_pattern.png','fig_eigenvalues.png',
        'fig_ode_convergence.png','fig_ode_solution.png','fig_heat_convergence.png',
        'fig_heat_solution.png','fig_work_accuracy.png','fig_symbol_analysis.png']
print('\n' + '='*55)
print('  All figures generated:')
for f in figs:
    sz = os.path.getsize(f)//1024 if os.path.exists(f) else 0
    print(f'  {"✓" if sz else "✗"} {f}  ({sz} KB)')
print('='*55)
print('Copy all PNGs to your LaTeX folder and compile with:')
print('  pdflatex wavelet_de_solution.tex  (×3, with bibtex between)')